# Definining cohort

**The goal of this notebook is to identify patients with metastatic renal cell carcinoma who received first-line combination immunotherapy (eg., ipilimumab + nivolumab) or single-agent antiangiogenic therapy (eg., pazopanib or sunitnib).**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## 1. Combination immunotherapy

In [3]:
therapy = pd.read_csv('../data/LineOfTherapy.csv')

In [4]:
therapy.head(5)

,PatientID,LineName,LineNumber,LineSetting,RegimenClass,IsMaintenanceTherapy,EnhancedCohort,StartDate,EndDate
0,F086DD8EA55B3,Cabozantinib,1,METASTATIC,NaN,False,MetastaticRCC,2018-04-25,2018-06-28
1,FDACCDA0762EB,Nivolumab,1,METASTATIC,NaN,False,MetastaticRCC,2017-05-04,2021-04-13
2,F42ACFD609CE6,Pazopanib,1,METASTATIC,NaN,False,MetastaticRCC,2015-05-08,2015-10-13
3,F42ACFD609CE6,Temsirolimus,2,METASTATIC,NaN,False,MetastaticRCC,2015-10-14,2015-12-03
4,F739F5568C0A1,Temsirolimus,1,METASTATIC,NaN,False,MetastaticRCC,2015-09-08,2015-09-25


In [5]:
therapy.query('LineNumber == 1').LineSetting.value_counts().head(20)

LineSetting
METASTATIC    8337
Name: count, dtype: int64

In [6]:
therapy.query('LineNumber == 1').IsMaintenanceTherapy.value_counts().head(20)

IsMaintenanceTherapy
False    8337
Name: count, dtype: int64

In [7]:
therapy.query('LineNumber == 1').LineName.value_counts().head(20)

LineName
Pazopanib                         1925
Sunitinib                         1895
Ipilimumab,Nivolumab              1011
Axitinib,Pembrolizumab             569
Temsirolimus                       509
Clinical Study Drug                451
Nivolumab                          371
Cabozantinib                       339
Axitinib                           189
Pembrolizumab                      158
Everolimus                         120
Cabozantinib,Nivolumab             102
Sorafenib                           88
Bevacizumab                         52
Aldesleukin                         47
Carboplatin,Paclitaxel              23
Gemcitabine,Sunitinib               21
Bevacizumab,Interferon Alfa-2B      19
Lenvatinib,Pembrolizumab            19
Avelumab,Axitinib                   17
Name: count, dtype: int64

In [8]:
ioio_df = (
    therapy
    .query('LineNumber == 1')
    .query('LineName == "Ipilimumab,Nivolumab"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'ioio'))

In [9]:
ioio_df.sample(3)

,PatientID,LineName,StartDate
16432,FC6962B24E53B,ioio,2021-10-04
10889,F1144D2D8A277,ioio,2020-02-05
16183,F46AAB200D781,ioio,2021-07-02


In [10]:
row_ID(ioio_df)

(1011, 1011)

## 2. Single-agent antiangiogenic therapy

In [11]:
tki_df = (
    therapy
    .query('LineNumber == 1')
    .query('LineName == "Pazopanib" or LineName == "Sunitinib"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'tki'))

In [12]:
tki_df.sample(3)

,PatientID,LineName,StartDate
12977,F182D5BDE75E0,tki,2017-07-12
9135,F9F80DC92F95F,tki,2015-05-11
5322,FADC2E195CDF4,tki,2014-12-15


In [13]:
row_ID(tki_df)

(3820, 3820)

## 3. Combine dataframes and export to csv 

In [14]:
firstline_ioio_tki_index = pd.concat([ioio_df, tki_df], axis = 0)

In [15]:
firstline_ioio_tki_index.sample(5)

,PatientID,LineName,StartDate
7672,F521B91BF37BE,tki,2014-12-28
1237,F79A74FE1A331,tki,2015-11-03
15543,F9BFFE9A5B992,tki,2014-10-02
3271,F111209DA0A13,tki,2012-01-05
2414,F04BAE26E523A,ioio,2019-12-03


In [16]:
row_ID(firstline_ioio_tki_index)

(4831, 4831)

In [17]:
firstline_ioio_tki_index.to_csv('../outputs/ioio_tki_index.csv', index = False)